In [1]:
import pandas as pd
import sqlite3

## create a connections with database

In [2]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

## using only one query for each of the groups, create two dataframes: test_tesults and contor_results with the columns time and avg_diff and only two rows

In [3]:
have_before = pd.read_sql('''
SELECT uid, time
FROM
(SELECT uid,
    CASE
        WHEN first_commit_ts <= first_view_ts THEN 'before'
        ELSE 'after' END as time
FROM control t JOIN deadlines d
    ON t.labname = d.labs
WHERE d.labs <> 'project1')

''', conn)
have_before

,uid,time
0,user_12,before
1,user_13,before
2,user_15,before
3,user_16,before
4,user_2,before
...,...,...
61,user_29,after
62,user_4,after
63,user_6,after
64,user_7,after


In [4]:
test_result = pd.read_sql('''
SELECT 
    CASE
        WHEN first_commit_ts <= first_view_ts THEN 'before'
        ELSE 'after'
    END as time,
    AVG((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600) AS avg_diff
FROM test t JOIN deadlines d
    ON t.labname = d.labs
WHERE d.labs <> 'project1' AND 
uid IN (SELECT uid
        FROM
        (SELECT uid,
            CASE
                WHEN first_commit_ts <= first_view_ts THEN 'before'
                ELSE 'after' END as time
        FROM test t JOIN deadlines d
            ON t.labname = d.labs
        WHERE d.labs <> 'project1')
        GROUP BY uid
        HAVING COUNT(DISTINCT time) = 2)
GROUP BY time
''', conn)

test_result

,time,avg_diff
0,after,-104.6000
1,before,-60.5625


In [5]:
control_result = pd.read_sql('''
SELECT 
    CASE
        WHEN first_commit_ts < first_view_ts THEN 'before'
        ELSE 'after'
    END as time,
    AVG((strftime('%s', c.first_commit_ts) - d.deadlines) / 3600) AS avg_diff
FROM control c JOIN deadlines d
    ON c.labname = d.labs
WHERE d.labs <> 'project1' AND
    uid IN (SELECT uid
        FROM
            (SELECT uid,
                CASE
                    WHEN first_commit_ts < first_view_ts THEN 'before'
                    ELSE 'after' END as time
            FROM control t JOIN deadlines d
                ON t.labname = d.labs
            WHERE d.labs <> 'project1')
        GROUP BY uid
        HAVING COUNT(DISTINCT time) = 2)
GROUP BY time
''', conn)

control_result

,time,avg_diff
0,after,-117.636364
1,before,-99.464286


## close the connection

In [6]:
conn.close()

In [ ]:
control_result

,time,avg_diff
0,after,-117.636364
1,before,-99.464286
